# 数据漂移检测教程

> **前置知识**: Python基础、统计学基础（均值、方差、分布）、监控概念（参见 03_Monitoring_tutorial）
>
> **学习目标**: 深入理解数据漂移的类型和检测方法

---

## 为什么需要漂移检测？

```
模型效果下降的常见原因:
┌─────────────────────────────────────────────────────────────┐
│  训练数据 ≠ 生产数据                                        │
│  - 用户群体变化（年龄、地区、偏好）                         │
│  - 季节性变化（节假日、促销活动）                           │
│  - 外部事件（疫情、政策变化）                               │
│  - 数据采集方式变化                                         │
└─────────────────────────────────────────────────────────────┘

漂移检测的价值:
┌─────────────────────────────────────────────────────────────┐
│  及时发现 → 在效果严重下降前预警                            │
│  定位问题 → 知道是哪些特征发生了变化                        │
│  触发重训练 → 自动化模型更新流程                            │
└─────────────────────────────────────────────────────────────┘
```

## 本教程内容

1. **漂移类型** - 数据漂移、概念漂移、标签漂移
2. **统计检测方法** - KS检验、PSI、JS散度
3. **在线漂移检测** - 滑动窗口实时检测
4. **多特征漂移检测** - 批量检测多个特征

In [ ]:
# ============================================================
# 环境准备
# ============================================================
import numpy as np
from scipy import stats
from collections import deque
from typing import Dict, List
import matplotlib.pyplot as plt

# 设置随机种子，确保结果可复现
np.random.seed(42)

print("=" * 50)
print("环境准备完成")
print("=" * 50)
print("本教程使用的库:")
print("  - numpy: 数值计算")
print("  - scipy.stats: 统计检验")
print("  - matplotlib: 可视化")

## 1. 漂移类型

**核心概念**: 漂移是指数据分布随时间发生变化

```
漂移类型分类:
┌─────────────────────────────────────────────────────────────┐
│                                                             │
│  数据漂移 (Covariate Shift)                                 │
│  - P(X) 发生变化                                            │
│  - 输入特征的分布改变                                        │
│  - 例: 用户年龄分布从25-35变成18-25                         │
│                                                             │
│  概念漂移 (Concept Drift)                                   │
│  - P(Y|X) 发生变化                                          │
│  - 特征与标签的关系改变                                      │
│  - 例: 疫情期间，同样的用户行为对应不同的购买意图            │
│                                                             │
│  标签漂移 (Label Drift)                                     │
│  - P(Y) 发生变化                                            │
│  - 标签的分布改变                                            │
│  - 例: 欺诈率从1%上升到5%                                   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

**检测难度**: 数据漂移 < 标签漂移 < 概念漂移

In [ ]:
# ============================================================
# 生成模拟数据演示漂移
# ============================================================
# 创建三组数据来演示漂移检测：
# 1. 参考数据（训练时的数据分布）
# 2. 无漂移数据（与参考数据相同分布）
# 3. 有漂移数据（分布发生变化）

# 参考数据：标准正态分布 N(0, 1)
reference_data = np.random.normal(0, 1, 1000)

# 无漂移数据：相同分布 N(0, 1)
no_drift_data = np.random.normal(0, 1, 1000)

# 有漂移数据：均值和方差都发生变化 N(0.5, 1.2)
drift_data = np.random.normal(0.5, 1.2, 1000)

# 可视化三组数据的分布
fig, axes = plt.subplots(1, 3, figsize=(12, 3))

axes[0].hist(reference_data, bins=30, alpha=0.7, color='blue')
axes[0].set_title('Reference Data\n(Training Distribution)')
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Frequency')

axes[1].hist(no_drift_data, bins=30, alpha=0.7, color='green')
axes[1].set_title('No Drift Data\n(Same Distribution)')
axes[1].set_xlabel('Value')

axes[2].hist(drift_data, bins=30, alpha=0.7, color='red')
axes[2].set_title('Drift Data\n(Mean & Std Changed)')
axes[2].set_xlabel('Value')

plt.tight_layout()
plt.show()

print("=" * 50)
print("数据统计对比")
print("=" * 50)
print(f"{'数据集':<15} {'均值':<10} {'标准差':<10}")
print("-" * 50)
print(f"{'参考数据':<15} {np.mean(reference_data):<10.4f} {np.std(reference_data):<10.4f}")
print(f"{'无漂移数据':<15} {np.mean(no_drift_data):<10.4f} {np.std(no_drift_data):<10.4f}")
print(f"{'有漂移数据':<15} {np.mean(drift_data):<10.4f} {np.std(drift_data):<10.4f}")

## 2. 统计检测方法

**核心方法对比**:

```
┌─────────────────────────────────────────────────────────────┐
│  方法              原理                    适用场景          │
│  ────────────      ────────────            ────────────      │
│  KS检验            比较累积分布函数        连续特征          │
│  PSI               比较区间占比变化        分布稳定性        │
│  JS散度            比较概率分布差异        概率分布          │
└─────────────────────────────────────────────────────────────┘
```

**选择建议**:
- **KS检验**: 最常用，适合连续特征，有统计学意义
- **PSI**: 业界标准，解释性强，有明确阈值
- **JS散度**: 对称性好，适合概率分布比较

In [ ]:
# ============================================================
# 漂移检测器类 - 实现多种检测方法
# ============================================================

class DriftDetector:
    """
    漂移检测器
    
    支持的检测方法:
    - KS检验 (Kolmogorov-Smirnov): 比较累积分布函数
    - PSI (Population Stability Index): 比较区间占比
    - JS散度 (Jensen-Shannon Divergence): 比较概率分布
    """
    
    @staticmethod
    def ks_test(reference, current, threshold=0.05):
        """
        KS检验 (Kolmogorov-Smirnov Test)
        
        原理:
        1. 计算两个样本的累积分布函数(CDF)
        2. 找到两个CDF之间的最大垂直距离
        3. 如果距离大于临界值，认为分布不同
        
        参数:
            reference: 参考数据（训练集）
            current: 当前数据（线上数据）
            threshold: p值阈值，小于此值认为有漂移
            
        返回:
            statistic: KS统计量，范围[0,1]，越大差异越大
            p_value: p值，越小越可能有漂移
        """
        stat, p_value = stats.ks_2samp(reference, current)
        return {
            'method': 'KS Test',
            'statistic': stat,
            'p_value': p_value,
            'drift_detected': p_value < threshold
        }
    
    @staticmethod
    def psi(reference, current, bins=10):
        """
        PSI (Population Stability Index)
        
        原理:
        1. 将数据分成N个区间（bins）
        2. 计算每个区间在两个分布中的占比
        3. PSI = Σ (actual% - expected%) × ln(actual% / expected%)
        
        PSI解读:
        - < 0.1: 无显著变化
        - 0.1 ~ 0.2: 轻微变化，需关注
        - > 0.2: 显著变化，需要行动
        """
        # 计算分位数边界（基于参考数据）
        ref_hist, edges = np.histogram(reference, bins=bins)
        cur_hist, _ = np.histogram(current, bins=edges)
        
        # 计算每个区间的占比（加小值避免除零）
        ref_pct = ref_hist / len(reference) + 1e-6
        cur_pct = cur_hist / len(current) + 1e-6
        
        # 计算PSI
        psi_val = np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct))
        
        return {
            'method': 'PSI',
            'value': psi_val,
            'drift_detected': psi_val > 0.2,
            'severity': 'none' if psi_val < 0.1 else ('minor' if psi_val < 0.2 else 'major')
        }
    
    @staticmethod
    def js_divergence(reference, current, bins=10):
        """
        JS散度 (Jensen-Shannon Divergence)
        
        原理:
        1. 计算两个分布的概率密度
        2. JS = 0.5 * KL(P||M) + 0.5 * KL(Q||M)，其中 M = (P+Q)/2
        3. JS散度是对称的，范围[0, ln(2)]
        
        优点: 对称性好，有界，适合概率分布比较
        """
        # 计算直方图（概率密度）
        ref_hist, edges = np.histogram(reference, bins=bins, density=True)
        cur_hist, _ = np.histogram(current, bins=edges, density=True)
        
        # 避免零值
        ref_hist = ref_hist + 1e-10
        cur_hist = cur_hist + 1e-10
        
        # 计算中间分布
        m = 0.5 * (ref_hist + cur_hist)
        
        # 计算JS散度
        js = 0.5 * (stats.entropy(ref_hist, m) + stats.entropy(cur_hist, m))
        
        return {
            'method': 'JS Divergence',
            'value': js,
            'drift_detected': js > 0.1
        }

print("DriftDetector 类定义完成")
print("支持的方法: ks_test, psi, js_divergence")

In [ ]:
# ============================================================
# 测试各种检测方法
# ============================================================
# 使用三种方法分别检测无漂移数据和有漂移数据

detector = DriftDetector()

print("=" * 60)
print("无漂移数据检测结果")
print("=" * 60)
ks_result = detector.ks_test(reference_data, no_drift_data)
psi_result = detector.psi(reference_data, no_drift_data)
js_result = detector.js_divergence(reference_data, no_drift_data)

print(f"KS检验:  统计量={ks_result['statistic']:.4f}, p值={ks_result['p_value']:.4f}, 漂移={'是' if ks_result['drift_detected'] else '否'}")
print(f"PSI:     值={psi_result['value']:.4f}, 严重程度={psi_result['severity']}, 漂移={'是' if psi_result['drift_detected'] else '否'}")
print(f"JS散度:  值={js_result['value']:.4f}, 漂移={'是' if js_result['drift_detected'] else '否'}")

print("\n" + "=" * 60)
print("有漂移数据检测结果")
print("=" * 60)
ks_result = detector.ks_test(reference_data, drift_data)
psi_result = detector.psi(reference_data, drift_data)
js_result = detector.js_divergence(reference_data, drift_data)

print(f"KS检验:  统计量={ks_result['statistic']:.4f}, p值={ks_result['p_value']:.6f}, 漂移={'是 ⚠️' if ks_result['drift_detected'] else '否'}")
print(f"PSI:     值={psi_result['value']:.4f}, 严重程度={psi_result['severity']}, 漂移={'是 ⚠️' if psi_result['drift_detected'] else '否'}")
print(f"JS散度:  值={js_result['value']:.4f}, 漂移={'是 ⚠️' if js_result['drift_detected'] else '否'}")

print("\n结论: 三种方法都能正确检测到漂移数据")

## 3. 在线漂移检测

**核心概念**: 实时监控数据流，及时发现漂移

```
在线检测 vs 离线检测:
┌─────────────────────────────────────────────────────────────┐
│  离线检测                      在线检测                      │
│  ──────────                    ──────────                    │
│  批量处理                      实时处理                      │
│  定期执行（每天/每周）         持续监控                      │
│  延迟发现                      及时发现                      │
│  资源消耗低                    资源消耗高                    │
└─────────────────────────────────────────────────────────────┘
```

**滑动窗口方法**:
- 维护一个固定大小的窗口
- 新数据进入时，旧数据移出
- 比较窗口内数据与基线的差异

In [ ]:
# ============================================================
# 在线漂移检测器 - 滑动窗口方法
# ============================================================

class OnlineDriftDetector:
    """
    在线漂移检测器（滑动窗口）
    
    原理:
    1. 维护一个固定大小的滑动窗口
    2. 新数据进入时，旧数据移出
    3. 计算窗口内数据与基线的Z分数
    4. Z分数超过阈值时判定为漂移
    
    适用场景: 实时数据流监控
    """
    
    def __init__(self, window_size=100, threshold=2.0):
        """
        参数:
            window_size: 滑动窗口大小
            threshold: Z分数阈值（通常2.0对应95%置信度）
        """
        self.window_size = window_size
        self.threshold = threshold
        self.window = deque(maxlen=window_size)  # 自动移除旧数据
        self.baseline_mean = None
        self.baseline_std = None
    
    def set_baseline(self, data):
        """设置基线（使用训练数据的统计量）"""
        self.baseline_mean = np.mean(data)
        self.baseline_std = np.std(data)
    
    def update(self, value):
        """
        更新窗口并检测漂移
        
        返回:
            drift_detected: 是否检测到漂移
            z_score: 当前Z分数
        """
        self.window.append(value)
        
        # 窗口未满时不检测
        if len(self.window) < self.window_size:
            return False, 0
        
        # 计算窗口均值的Z分数
        # Z = (窗口均值 - 基线均值) / (基线标准差 / sqrt(窗口大小))
        current_mean = np.mean(list(self.window))
        z_score = abs(current_mean - self.baseline_mean) / (self.baseline_std / np.sqrt(self.window_size))
        
        return z_score > self.threshold, z_score

# ============================================================
# 模拟在线检测
# ============================================================
online_detector = OnlineDriftDetector(window_size=50, threshold=2.0)
online_detector.set_baseline(reference_data)

# 模拟数据流：前500个正常，后500个漂移
stream = np.concatenate([
    np.random.normal(0, 1, 500),   # 正常数据
    np.random.normal(1, 1, 500)    # 漂移数据（均值从0变为1）
])

# 检测漂移
drift_points = []
z_scores = []

for i, val in enumerate(stream):
    drift, z = online_detector.update(val)
    z_scores.append(z)
    if drift and len(drift_points) == 0:  # 记录首次检测到漂移的位置
        drift_points.append(i)

print("=" * 50)
print("在线漂移检测结果")
print("=" * 50)
print(f"数据流长度: {len(stream)}")
print(f"漂移开始位置: 500")
print(f"首次检测到漂移: {drift_points[0] if drift_points else '未检测到'}")
print(f"检测延迟: {drift_points[0] - 500 if drift_points else 'N/A'} 个样本")

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 数据流
axes[0].plot(stream, alpha=0.5, linewidth=0.5)
axes[0].axvline(x=500, color='r', linestyle='--', label='Drift Start')
if drift_points:
    axes[0].axvline(x=drift_points[0], color='g', linestyle='--', label='First Detection')
axes[0].set_title('Data Stream')
axes[0].set_xlabel('Sample Index')
axes[0].set_ylabel('Value')
axes[0].legend()

# Z分数
axes[1].plot(z_scores, linewidth=0.8)
axes[1].axhline(y=2.0, color='r', linestyle='--', label='Threshold (Z=2)')
axes[1].axvline(x=500, color='orange', linestyle='--', alpha=0.5, label='Drift Start')
axes[1].set_title('Z-Score Over Time')
axes[1].set_xlabel('Sample Index')
axes[1].set_ylabel('Z-Score')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. 多特征漂移检测

**核心概念**: 实际场景中需要同时监控多个特征的漂移

```
多特征漂移检测策略:
┌─────────────────────────────────────────────────────────────┐
│  1. 逐特征检测: 对每个特征单独检测                          │
│  2. 汇总判断: 根据漂移特征比例判断整体漂移                  │
│  3. 重要性加权: 对重要特征给予更高权重                      │
└─────────────────────────────────────────────────────────────┘

判断标准:
- 漂移特征比例 > 30%: 整体漂移
- 关键特征漂移: 即使比例低也需关注
```

In [ ]:
# ============================================================
# 多特征漂移检测器
# ============================================================

class MultiFeatureDriftDetector:
    """
    多特征漂移检测器
    
    功能:
    1. 对每个特征单独进行漂移检测
    2. 汇总各特征的检测结果
    3. 根据漂移特征比例判断整体漂移
    """
    
    def __init__(self, reference_data, feature_names):
        """
        参数:
            reference_data: 参考数据，shape=(n_samples, n_features)
            feature_names: 特征名称列表
        """
        self.reference = reference_data
        self.feature_names = feature_names
    
    def detect(self, current_data, threshold_ratio=0.3):
        """
        检测多特征漂移
        
        参数:
            current_data: 当前数据，shape=(n_samples, n_features)
            threshold_ratio: 漂移特征比例阈值
            
        返回:
            feature_results: 各特征的检测结果
            drift_count: 漂移特征数量
            drift_ratio: 漂移特征比例
            overall_drift: 是否整体漂移
        """
        results = {}
        drift_count = 0
        
        # 对每个特征单独检测
        for i, name in enumerate(self.feature_names):
            ref_col = self.reference[:, i]
            cur_col = current_data[:, i]
            
            # 使用PSI检测
            result = DriftDetector.psi(ref_col, cur_col)
            results[name] = result
            
            if result['drift_detected']:
                drift_count += 1
        
        return {
            'feature_results': results,
            'drift_count': drift_count,
            'drift_ratio': drift_count / len(self.feature_names),
            'overall_drift': drift_count > len(self.feature_names) * threshold_ratio
        }

# ============================================================
# 测试多特征漂移检测
# ============================================================

# 创建4个特征的数据
feature_names = ['feature_1', 'feature_2', 'feature_3', 'feature_4']

# 参考数据：标准正态分布
ref_multi = np.random.randn(1000, 4)

# 当前数据：部分特征发生漂移
cur_multi = np.random.randn(1000, 4)
cur_multi[:, 0] += 1    # 第1个特征：均值偏移
cur_multi[:, 2] *= 2    # 第3个特征：方差变大

# 创建检测器并检测
multi_detector = MultiFeatureDriftDetector(ref_multi, feature_names)
result = multi_detector.detect(cur_multi)

print("=" * 60)
print("多特征漂移检测结果")
print("=" * 60)
print(f"整体漂移: {'是 ⚠️' if result['overall_drift'] else '否 ✓'}")
print(f"漂移特征数: {result['drift_count']} / {len(feature_names)}")
print(f"漂移比例: {result['drift_ratio']:.2%}")

print("\n各特征详情:")
print("-" * 60)
print(f"{'特征名':<15} {'PSI值':<12} {'严重程度':<10} {'漂移'}")
print("-" * 60)
for name, res in result['feature_results'].items():
    drift_mark = '⚠️' if res['drift_detected'] else '✓'
    print(f"{name:<15} {res['value']:<12.4f} {res['severity']:<10} {drift_mark}")

## 总结

本教程介绍了数据漂移检测的核心方法：

```
核心概念回顾:
┌─────────────────────────────────────────────────────────────┐
│  漂移类型:                                                   │
│  ├── 数据漂移: P(X) 变化 - 输入特征分布改变                 │
│  ├── 概念漂移: P(Y|X) 变化 - 特征与标签关系改变             │
│  └── 标签漂移: P(Y) 变化 - 标签分布改变                     │
│                                                             │
│  检测方法:                                                   │
│  ├── KS检验: 比较累积分布函数                               │
│  ├── PSI: 比较区间占比变化                                  │
│  ├── JS散度: 比较概率分布差异                               │
│  └── 在线检测: 滑动窗口实时监控                             │
└─────────────────────────────────────────────────────────────┘
```

### 方法选择指南

| 方法 | 适用场景 | 阈值 | 优点 |
|:-----|:---------|:-----|:-----|
| KS检验 | 连续特征 | p < 0.05 | 有统计学意义 |
| PSI | 分布稳定性 | > 0.2 | 解释性强 |
| JS散度 | 概率分布 | > 0.1 | 对称性好 |
| 在线检测 | 实时监控 | Z > 2.0 | 及时发现 |

### 最佳实践

| 实践 | 说明 |
|:-----|:-----|
| 定期检测 | 每天/每周批量检测 |
| 实时监控 | 关键特征使用在线检测 |
| 多方法结合 | 使用多种方法交叉验证 |
| 关注重要特征 | 对模型影响大的特征优先监控 |
| 设置告警 | 漂移检测到后自动通知 |

### 下一步

- 学习 [05_ABTesting_tutorial](05_ABTesting_tutorial.ipynb) 了解 A/B 测试
- 学习 [06_AutoRetraining_tutorial](06_AutoRetraining_tutorial.ipynb) 了解自动重训练